In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

In [2]:
current_directory = Path.cwd().parent
discharge_directory = Path.joinpath(current_directory, "Data Start Point")

points_of_interest_file_name = Path.joinpath(discharge_directory, "points_of_interest.json")
battery_state_file_name = Path.joinpath(discharge_directory, "battery_state.csv")
compiled_file_name = Path.joinpath(discharge_directory, "final_cleaned_compiled (with SoC).csv")

output_file_name = Path.joinpath(discharge_directory, "battery_constants.json")

In [3]:
compiled_df = pd.read_csv(compiled_file_name)
print(compiled_df.head())

   Translated Time  Translated Time (minutes)  \
0            0.000                   0.000000   
1            0.942                   0.000016   
2            1.880                   0.000031   
3            2.819                   0.000047   
4            3.757                   0.000063   

   ADS Reading 1 (Smoothed and Offset)  Calibrated Current 1  \
0                              32755.0             -0.031158   
1                              32746.0             -0.053594   
2                              32737.0             -0.076032   
3                              32755.0             -0.031158   
4                              32737.0             -0.076032   

   ADS Reading 2 (Smoothed and Offset)  Calibrated Current 2  \
0                              32730.0             -0.093485   
1                              32722.5             -0.112186   
2                              32722.0             -0.113433   
3                              32722.0             -0.113433   


In [5]:
sensor_draw_estimate_Ah = 0.081
sensor_draw_estimate_coulombs = sensor_draw_estimate_Ah * 3600

total_charge = compiled_df['Charge Change'].sum() - sensor_draw_estimate_coulombs
total_Ah = (total_charge / 3600)

print(f"Total Charge: {total_charge:.4f} Coulombs")
print(f"Total Ah: {total_Ah:.4f} Ah")

Total Charge: 143493.5685 Coulombs
Total Ah: 39.8593 Ah


In [6]:
points_of_interest = json.load(open(points_of_interest_file_name, 'r'))
before_close = points_of_interest['before_close']
before_open = points_of_interest['before_open']

In [7]:
STEP_BACK = 10
SAMPLING_RANGE = 50

sd_voltage_array = []

for index in before_close['index'][:-1]: # Exclude the last point which have the highest noise from shutting down
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].mean()
    sd_voltage = compiled_df.loc[start_index:end_index, 'Calibrated Voltage'].std()
    sd_voltage_array.append(sd_voltage)
    print(f"Mean Voltage: {mean_voltage:.4f} V, SD Voltage: {sd_voltage:.4f} V")

sigma_v = np.mean(sd_voltage_array)
print(f"Mean Standard Deviations: {sigma_v:.4f} V")

Mean Voltage: 52.3665 V, SD Voltage: 0.0008 V
Mean Voltage: 52.6363 V, SD Voltage: 0.0008 V
Mean Voltage: 52.3263 V, SD Voltage: 0.0010 V
Mean Voltage: 52.2214 V, SD Voltage: 0.0007 V
Mean Voltage: 52.1886 V, SD Voltage: 0.0013 V
Mean Voltage: 51.9050 V, SD Voltage: 0.0007 V
Mean Standard Deviations: 0.0009 V


In [8]:
# During circuit open, where current = 0, almost no noise.
# Need to use a realisitic point where current is flowing to estimate the noise in current measurement.

STEP_FORWARD = 10
SAMPLING_RANGE = 500

sd_zero_current_array = []

for index in before_open['index']:
    start_index = min(len(compiled_df) - 1, index + STEP_FORWARD)
    end_index = min(len(compiled_df) - 1, start_index + SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    sd_zero_current_array.append(sd_current)
    
    print(f"Mean Current: {mean_current:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_zero_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 23.2812 A, SD Current: 6.2053 A
Mean Current: 24.0034 A, SD Current: 5.1655 A
Mean Current: 26.0350 A, SD Current: 0.0305 A
Mean Current: 22.6282 A, SD Current: 7.0293 A
Mean Current: 22.6136 A, SD Current: 6.8123 A
Mean Current: 22.4808 A, SD Current: 6.9442 A
Mean Current: 22.1019 A, SD Current: 7.2918 A
Mean Standard Deviations: 5.6398 A


In [9]:
STEP_BACK = 10
SAMPLING_RANGEs = [100, 200, 300, 400]

for sampling_range in SAMPLING_RANGEs:
    sd_current_array = []

    for index in before_open['index']:
        end_index = max(0, index - STEP_BACK)
        start_index = max(0, end_index - sampling_range)
        mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
        mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
        
        sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
        sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
        
        mean_current = (mean_current_1 + mean_current_2) / 2
        sd_current = (sd_current_1 + sd_current_2) / 2
        
        sd_current_array.append(sd_current)
        #print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current_1:.4f} A")
    print(f"Mean Standard Deviations ({sampling_range}) samples: {np.mean(sd_current_array):.4f} A")

Mean Standard Deviations (100) samples: 0.0278 A
Mean Standard Deviations (200) samples: 0.0280 A
Mean Standard Deviations (300) samples: 0.0292 A
Mean Standard Deviations (400) samples: 0.0295 A


In [10]:
STEP_BACK = 10
SAMPLING_RANGE = 200

sd_current_array = []

for index in before_open['index']:
    end_index = max(0, index - STEP_BACK)
    start_index = max(0, end_index - SAMPLING_RANGE)
    mean_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].mean()
    mean_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].mean()
    
    sd_current_1 = compiled_df.loc[start_index:end_index, 'Calibrated Current 1'].std()
    sd_current_2 = compiled_df.loc[start_index:end_index, 'Calibrated Current 2'].std()
    
    mean_current = (mean_current_1 + mean_current_2) / 2
    sd_current = (sd_current_1 + sd_current_2) / 2
    
    sd_current_array.append(sd_current)
    print(f"Mean Current: {mean_current_1:.4f} A, SD Current: {sd_current:.4f} A")
    
sigma_i = np.mean(sd_current_array)
print(f"Mean Standard Deviations: {sigma_i:.4f} A")

Mean Current: 25.8872 A, SD Current: 0.0251 A
Mean Current: 26.4771 A, SD Current: 0.0274 A
Mean Current: 0.0097 A, SD Current: 0.0222 A
Mean Current: 25.5439 A, SD Current: 0.0300 A
Mean Current: 25.4589 A, SD Current: 0.0277 A
Mean Current: 25.5521 A, SD Current: 0.0314 A
Mean Current: 25.2112 A, SD Current: 0.0319 A
Mean Standard Deviations: 0.0280 A


In [11]:
sigma_kcl = np.sqrt(4* sigma_i**2)
print(f"Sigma KCL: {sigma_kcl:.4f} A")

Sigma KCL: 0.0559 A


In [12]:
battery_state = pd.read_csv(battery_state_file_name)
soc_factor = len(str(battery_state['SoC'].iloc[1]).split('.')[1])

print(soc_factor)

3


In [14]:
battery_constant = {
    'Q_total': total_Ah,
    'sigma_v': sigma_v,
    'sigma_i': sigma_i,
    'sigma_kcl': sigma_kcl,
    'interval_factor': soc_factor
}

print(f"Battery Constants: {json.dumps(battery_constant, indent=4)}")

Battery Constants: {
    "Q_total": 39.859324569606194,
    "sigma_v": 0.0008893445709016291,
    "sigma_i": 0.027951736148341476,
    "sigma_kcl": 0.05590347229668295,
    "interval_factor": 3
}


In [15]:
json.dump(battery_constant, open(output_file_name, 'w'), indent=4)
print(f"Battery constants saved to {output_file_name}")

Battery constants saved to c:\Users\Kor\Documents\GitHub\Proa-II-Electrical-Setup\Calibration Data 2\Data\Data Start Point\battery_constants.json
